In [1]:
from google.cloud import bigquery
import pandas as pd
import requests

In [2]:
# Weather Archive API data frame: 2025 year sample
# Date: 2025-01-01 to 2025-12-31
# Location: London; latitude: 51.5; longitude: -0.1

url = 'https://archive-api.open-meteo.com/v1/archive'
params = {
	'latitude': 51.5,
	'longitude': -0.1,
	'start_date': '2025-01-01',
	'end_date': '2025-12-31',
	'hourly': [
        'temperature_2m',
        'wind_speed_10m',
        'wind_speed_100m',
        'wind_direction_10m',
        'wind_direction_100m',
        'wind_gusts_10m',
        'cloud_cover',
        'cloud_cover_low',
        'cloud_cover_mid',
        'cloud_cover_high',
        'shortwave_radiation',
        'direct_radiation',
        'diffuse_radiation',
        'direct_normal_irradiance',
        'global_tilted_irradiance',
        'terrestrial_radiation',
        'pressure_msl',
        'snowfall',
        'rain',
        'precipitation'
        ],
}

weather_a_data = requests.get(url, params=params).json()
weather_a_df = pd.DataFrame(weather_a_data['hourly'])

# coverting to datetime and rest index
weather_a_df['time'] = pd.to_datetime(weather_a_df['time'])
weather_a_df = weather_a_df.set_index('time')


print(weather_a_df.head())

                     temperature_2m  wind_speed_10m  wind_speed_100m  \
time                                                                   
2025-01-01 00:00:00            10.5            25.1             48.0   
2025-01-01 01:00:00            11.1            25.1             46.3   
2025-01-01 02:00:00            11.1            22.9             43.6   
2025-01-01 03:00:00            11.1            22.0             42.1   
2025-01-01 04:00:00            10.9            23.4             43.6   

                     wind_direction_10m  wind_direction_100m  wind_gusts_10m  \
time                                                                           
2025-01-01 00:00:00                 225                  228            55.1   
2025-01-01 01:00:00                 224                  225            58.3   
2025-01-01 02:00:00                 220                  222            60.5   
2025-01-01 03:00:00                 224                  225            61.9   
2025-01-01 04:0

In [3]:
weather_a_df.shape

(8760, 20)

In [4]:
weather_a_df = weather_a_df.rename(columns={
    'time': 'datetime',
    'temperature_2m': 'temperature_2m_c',
    'wind_speed_10m': 'wind_speed_10m_ms',
    'wind_speed_100m': 'wind_speed_100m_ms',
    'wind_direction_10m': 'wind_direction_10m_deg',
    'wind_direction_100m': 'wind_direction_100m_deg',
    'wind_gusts_10m': 'wind_gusts_10m_ms',
    'cloud_cover': 'cloud_cover_pct',
    'cloud_cover_low': 'cloud_cover_low_pct',
    'cloud_cover_mid': 'cloud_cover_mid_pct',
    'cloud_cover_high': 'cloud_cover_high_pct',
    'shortwave_radiation': 'shortwave_radiation_wm2',
    'direct_radiation': 'direct_radiation_wm2',
    'diffuse_radiation': 'diffuse_radiation_wm2',
    'direct_normal_irradiance': 'direct_normal_irradiance_wm2',
    'global_tilted_irradiance': 'global_tilted_irradiance_wm2',
    'terrestrial_radiation': 'terrestrial_radiation_wm2',
    'pressure_msl': 'pressure_msl_hpa',
    'snowfall': 'snowfall_cm',
    'rain': 'rain_mm',
    'precipitation': 'precipitation_mm'
})

In [5]:
# Upload to BQ on GCP
PROJECT = 'gridzero-489711'
DATASET ='historical_weather'
TABLE = '02_full_params_weather_2025'

table = f'{PROJECT}.{DATASET}.{TABLE}'

client = bigquery.Client(project=PROJECT)

write_mode = 'WRITE_TRUNCATE'
job_config = bigquery.LoadJobConfig(write_disposition=write_mode)

job = client.load_table_from_dataframe(weather_a_df, table, job_config=job_config)
result = job.result()

/Users/madeleine/.pyenv/versions/3.12.9/envs/gridzero/lib/python3.12/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


In [6]:
# convert timestamp to half-hourly (hhr) for joining
weather_a_df_hhr = weather_a_df.resample('30min').ffill()

print(weather_a_df_hhr.head())
print(weather_a_df_hhr.shape)

                     temperature_2m_c  wind_speed_10m_ms  wind_speed_100m_ms  \
time                                                                           
2025-01-01 00:00:00              10.5               25.1                48.0   
2025-01-01 00:30:00              10.5               25.1                48.0   
2025-01-01 01:00:00              11.1               25.1                46.3   
2025-01-01 01:30:00              11.1               25.1                46.3   
2025-01-01 02:00:00              11.1               22.9                43.6   

                     wind_direction_10m_deg  wind_direction_100m_deg  \
time                                                                   
2025-01-01 00:00:00                     225                      228   
2025-01-01 00:30:00                     225                      228   
2025-01-01 01:00:00                     224                      225   
2025-01-01 01:30:00                     224                      225   
2025-01

In [7]:
# Upload to BQ on GCP
PROJECT = 'gridzero-489711'
DATASET ='historical_weather'
TABLE = '03_halfhourly_full_params_weather_2025'

table = f'{PROJECT}.{DATASET}.{TABLE}'

client = bigquery.Client(project=PROJECT)

write_mode = 'WRITE_TRUNCATE'
job_config = bigquery.LoadJobConfig(write_disposition=write_mode)

job = client.load_table_from_dataframe(weather_a_df_hhr, table, job_config=job_config)
result = job.result()

/Users/madeleine/.pyenv/versions/3.12.9/envs/gridzero/lib/python3.12/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


In [8]:
# Selected params only
cols = weather_a_df.columns.tolist()
print(cols)

cols_select = [
    'temperature_2m_c',
    'wind_speed_100m_ms',
    'wind_gusts_10m_ms',
    'cloud_cover_pct',
    'shortwave_radiation_wm2',
    'direct_radiation_wm2',
    'diffuse_radiation_wm2',
    'pressure_msl_hpa',
    'snowfall_cm',
    'rain_mm',
    'precipitation_mm'

]

['temperature_2m_c', 'wind_speed_10m_ms', 'wind_speed_100m_ms', 'wind_direction_10m_deg', 'wind_direction_100m_deg', 'wind_gusts_10m_ms', 'cloud_cover_pct', 'cloud_cover_low_pct', 'cloud_cover_mid_pct', 'cloud_cover_high_pct', 'shortwave_radiation_wm2', 'direct_radiation_wm2', 'diffuse_radiation_wm2', 'direct_normal_irradiance_wm2', 'global_tilted_irradiance_wm2', 'terrestrial_radiation_wm2', 'pressure_msl_hpa', 'snowfall_cm', 'rain_mm', 'precipitation_mm']


In [9]:
# selected hourly df
weather_a_df_select = weather_a_df[cols_select]
print(weather_a_df_select.head)

<bound method NDFrame.head of                      temperature_2m_c  wind_speed_100m_ms  wind_gusts_10m_ms  \
time                                                                           
2025-01-01 00:00:00              10.5                48.0               55.1   
2025-01-01 01:00:00              11.1                46.3               58.3   
2025-01-01 02:00:00              11.1                43.6               60.5   
2025-01-01 03:00:00              11.1                42.1               61.9   
2025-01-01 04:00:00              10.9                43.6               58.3   
...                               ...                 ...                ...   
2025-12-31 19:00:00               0.2                22.2               22.0   
2025-12-31 20:00:00              -0.3                22.0               22.3   
2025-12-31 21:00:00              -0.8                22.5               23.4   
2025-12-31 22:00:00              -0.9                23.3               23.8   
2025-12-31

In [10]:
# Upload to BQ on GCP
PROJECT = 'gridzero-489711'
DATASET ='historical_weather'
TABLE = '04_select_params_weather_2025'

table = f'{PROJECT}.{DATASET}.{TABLE}'

client = bigquery.Client(project=PROJECT)

write_mode = 'WRITE_TRUNCATE'
job_config = bigquery.LoadJobConfig(write_disposition=write_mode)

job = client.load_table_from_dataframe(weather_a_df_select, table, job_config=job_config)
result = job.result()

/Users/madeleine/.pyenv/versions/3.12.9/envs/gridzero/lib/python3.12/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


In [11]:
# selected half hourly df
weather_a_df_hhr_select = weather_a_df_hhr[cols_select]
print(weather_a_df_hhr_select.head)

<bound method NDFrame.head of                      temperature_2m_c  wind_speed_100m_ms  wind_gusts_10m_ms  \
time                                                                           
2025-01-01 00:00:00              10.5                48.0               55.1   
2025-01-01 00:30:00              10.5                48.0               55.1   
2025-01-01 01:00:00              11.1                46.3               58.3   
2025-01-01 01:30:00              11.1                46.3               58.3   
2025-01-01 02:00:00              11.1                43.6               60.5   
...                               ...                 ...                ...   
2025-12-31 21:00:00              -0.8                22.5               23.4   
2025-12-31 21:30:00              -0.8                22.5               23.4   
2025-12-31 22:00:00              -0.9                23.3               23.8   
2025-12-31 22:30:00              -0.9                23.3               23.8   
2025-12-31

In [12]:
# Upload to BQ on GCP
PROJECT = 'gridzero-489711'
DATASET ='historical_weather'
TABLE = '05_halfhourly_select_params_weather_2025'

table = f'{PROJECT}.{DATASET}.{TABLE}'

client = bigquery.Client(project=PROJECT)

write_mode = 'WRITE_TRUNCATE'
job_config = bigquery.LoadJobConfig(write_disposition=write_mode)

job = client.load_table_from_dataframe(weather_a_df_hhr_select, table, job_config=job_config)
result = job.result()

/Users/madeleine/.pyenv/versions/3.12.9/envs/gridzero/lib/python3.12/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


In [13]:
def fetch_weather(start_date, end_date, latitude=51.5, longitude=-0.1):

    '''fetch api data from open-meteo archive,
    returns selected parameters hourly for London (based on long and lat).
    Dates in string format.'''

    url = 'https://archive-api.open-meteo.com/v1/archive'

    selected_params = {
        'latitude': latitude,
        'longitude': longitude,
        'start_date': start_date,
        'end_date': end_date,
        'timezone': 'UTC',
        'hourly': [
            'temperature_2m',
            'wind_speed_100m',
            'wind_gusts_10m',
            'cloud_cover',
            'shortwave_radiation',
            'direct_radiation',
            'diffuse_radiation',
            'pressure_msl',
            'snowfall',
            'rain',
            'precipitation'
        ]
    }
    response = requests.get(url, params=selected_params).json()
    return pd.DataFrame(response['hourly'])

In [14]:
# fetch function test
df = fetch_weather('2025-02-10','2025-02-11')

In [15]:
def weather_preproc(df):
    ''' preprocess weather dataframe, resample, rename, check quality'''
    # datetime and set index
    df['time'] = pd.to_datetime(df['time'])
    df = df.set_index('time')

    # sample half hourly
    df = df.resample('30min').ffill()

    # reset index and rename to datetime
    df = df.reset_index()
    df = df.rename(columns={'time': 'datetime'})

    # rename columns w/ units
    df = df.rename(columns={
        'temperature_2m': 'temperature_2m_c',
        'wind_speed_100m': 'wind_speed_100m_ms',
        'wind_gusts_10m': 'wind_gusts_10m_ms',
        'cloud_cover': 'cloud_cover_pct',
        'shortwave_radiation': 'shortwave_radiation_wm2',
        'direct_radiation': 'direct_radiation_wm2',
        'diffuse_radiation': 'diffuse_radiation_wm2',
        'pressure_msl': 'pressure_msl_hpa',
        'snowfall': 'snowfall_cm',
        'rain': 'rain_mm',
        'precipitation': 'precipitation_mm'
    })

    return df


In [16]:
df_preproc = weather_preproc(df)

In [17]:
# checks
print(f'Shape: {df_preproc.shape}')
print(f'Nulls:\n{df_preproc.isnull().sum()}')
print(f'Duplicates: {df_preproc.duplicated().sum()}')

Shape: (95, 12)
Nulls:
datetime                   0
temperature_2m_c           0
wind_speed_100m_ms         0
wind_gusts_10m_ms          0
cloud_cover_pct            0
shortwave_radiation_wm2    0
direct_radiation_wm2       0
diffuse_radiation_wm2      0
pressure_msl_hpa           0
snowfall_cm                0
rain_mm                    0
precipitation_mm           0
dtype: int64
Duplicates: 0


In [20]:
from google.cloud import bigquery
import pandas as pd

def load_from_bigquery(PROJECT: str, DATASET: str, TABLE: str):
    '''
    Load data from BigQuery into a pandas DataFrame.

    Arguments:
        PROJECT: GCP project ID
        DATASET: BigQuery dataset name
        TABLE: BigQuery table name

    Returns: pandas DataFrame
    '''

    client = bigquery.Client(project=PROJECT)

    query = f'''
    SELECT *
    FROM `{PROJECT}.{DATASET}.{TABLE}`
    '''

    df = client.query(query).to_dataframe()

    rows, cols = df.shape
    print(f"Loaded {rows:,} rows and {cols} columns from {PROJECT}.{DATASET}.{TABLE}")

    return df

In [21]:
load_from_bigquery('gridzero-489711', 'merged_set', 'Fully_merged_dataset_2025')

/Users/madeleine/.pyenv/versions/3.12.9/envs/gridzero/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Loaded 17,519 rows and 23 columns from gridzero-489711.merged_set.Fully_merged_dataset_2025


,time,temperature_2m_c,wind_speed_100m_ms,wind_gusts_10m_ms,cloud_cover_pct,shortwave_radiation_wm2,direct_radiation_wm2,diffuse_radiation_wm2,pressure_msl_hpa,snowfall_cm,...,Fossil Oil,Hydro Pumped Storage,Hydro Run-of-river and poundage,Nuclear,Other,Solar,Wind Offshore,Wind Onshore,TotalOutput-MW,carbon_intensity
0,2025-01-01 00:00:00+00:00,10.5,48.0,55.1,100,0.0,0.0,0.0,1014.2,0.0,...,0.0,0.0,736.0,5065.0,183.0,0.0,11444.531,9113.028,31028.559,55
1,2025-01-01 00:30:00+00:00,10.5,48.0,55.1,100,0.0,0.0,0.0,1014.2,0.0,...,0.0,0.0,745.0,5063.0,290.0,0.0,11138.565,8969.868,31138.433,54
2,2025-01-01 01:00:00+00:00,11.1,46.3,58.3,100,0.0,0.0,0.0,1013.1,0.0,...,0.0,0.0,746.0,5056.0,333.0,0.0,10788.770,8931.922,30826.692,53
3,2025-01-01 01:30:00+00:00,11.1,46.3,58.3,100,0.0,0.0,0.0,1013.1,0.0,...,0.0,0.0,746.0,5060.0,238.0,0.0,10519.534,8810.976,30209.510,53
4,2025-01-01 02:00:00+00:00,11.1,43.6,60.5,100,0.0,0.0,0.0,1012.0,0.0,...,0.0,0.0,747.0,5056.0,277.0,0.0,10706.056,8456.386,29988.442,47
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17514,2025-12-31 21:00:00+00:00,-0.8,22.5,23.4,5,0.0,0.0,0.0,1023.1,0.0,...,0.0,4.0,378.0,4082.0,743.0,0.0,10803.097,5827.825,31071.922,<NA>
17515,2025-12-31 21:30:00+00:00,-0.8,22.5,23.4,5,0.0,0.0,0.0,1023.1,0.0,...,0.0,4.0,376.0,4086.0,669.0,0.0,10832.486,5820.693,30313.179,<NA>
17516,2025-12-31 22:00:00+00:00,-0.9,23.3,23.8,0,0.0,0.0,0.0,1021.9,0.0,...,0.0,4.0,361.0,4097.0,494.0,0.0,10937.104,6065.013,30201.117,<NA>
17517,2025-12-31 22:30:00+00:00,-0.9,23.3,23.8,0,0.0,0.0,0.0,1021.9,0.0,...,0.0,4.0,353.0,4094.0,253.0,0.0,11018.705,6001.891,30088.596,<NA>
